# dl_08 — UNet / UNet3+ Feature Map Visualization

This notebook opens up a trained wetland-classification model and shows, in pictures,
**what happens inside it** as a single aerial image patch passes through. It's written to
be readable even if you've never worked with deep learning.

### What is this model doing?
The model is a **U-Net** (here, the **UNet3+** variant) — a neural network for
*semantic segmentation*, i.e. assigning **every pixel** of an image a class. In this
project the classes are wetland types (**EMW** emergent marsh, **FSW** forested/shrub
wetland, **SSW** scrub-shrub wetland) plus **UPL** = upland (non-wetland). The model reads
a stack of georeferenced input layers — elevation, slope, aerial photos, canopy height,
etc. — and outputs a class for each pixel.

A U-Net has three parts, and this notebook visualizes each one:

- **Encoder** — repeatedly shrinks the image while pulling out features, moving from raw
  cues (edges, colors, textures) toward abstract concepts ("this region looks like marsh").
  Think of it as *zooming out* to understand context.
- **Bottleneck** — the smallest, most abstract representation, in the middle of the network.
- **Decoder** — rebuilds full resolution, combining that abstract understanding with the
  fine detail saved from the encoder, to draw sharp class boundaries. Think of it as
  *zooming back in* to place the labels precisely.

The "U" is literal — data flows down the left side, across the bottom, and up the right.
For this project's data (a 256 m × 256 m patch at 1 m resolution, depth-4 model):

```text
in: 256×256 patch, 26 layers                out: 256×256 map, 4 classes
  enc0 (256×256) ── skip ─────────────────────→ dec0 (256×256) → prediction
    enc1 (128×128) ── skip ──────────────→ dec1 (128×128)
      enc2 (64×64) ── skip ─────────→ dec2 (64×64)
        enc3 (32×32) ── skip ──→ dec3 (32×32)
                 bottleneck (16×16)
```

The horizontal **skip connections** carry fine detail straight across the U so the decoder
doesn't have to reconstruct it from the blurry bottleneck. (In **UNet3+**, every decoder
node actually receives skips from *all* encoder levels at once, not just its own — that's
the "3+".) For a written walk-through, see `UNet_Architecture_Overview.md` in this folder.

A **feature map** is one channel of a layer's output — a grayscale image where bright
pixels mean *"this learned pattern fired strongly here."* Early layers have feature maps
for simple things (edges, brightness); deep layers have feature maps for complex,
wetland-specific patterns. The plots below walk through these stage by stage, ending in the
final prediction.

### How it works
Runs **one** patch through the model and taps each stage with PyTorch *forward hooks*
(read-only — the model itself is never changed). Two ways to view each stage:

- **Top-variance channels** (default): the 1–3 most active channels — the ones actually carrying signal.
- **Mean activation map**: all channels averaged into one image — a clean "what does this level respond to" summary.

The bottleneck shows **both** side by side, and there's a whole-network overview.
Compute is tiny: one forward pass on a single patch (runs fine on CPU).

## 1. Imports & paths

Load the libraries and locate the project folders — `Models/` (trained checkpoints) and
`Data/Training_Data/R_Patches/` (the 256×256 image patches). Nothing model-specific
happens yet; this just wires things up.

In [5]:
import sys
import json
from pathlib import Path

import numpy as np
import torch
import rasterio
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap

# Notebook lives in Python_Code_Analysis/DL_Pipeline_v2 — make the dl_* modules importable.
SCRIPT_DIR = Path.cwd()
sys.path.insert(0, str(SCRIPT_DIR))

# Walk up to the project root (the dir that contains Models/ and Data/).
PROJECT_ROOT = SCRIPT_DIR
while not (PROJECT_ROOT / "Models").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

MODELS_DIR = PROJECT_ROOT / "Models"
PATCHES_DIR = PROJECT_ROOT / "Data" / "Training_Data" / "R_Patches"
STATS_DIR = PROJECT_ROOT / "Data" / "Training_Data"

from dl_model_utils import load_model  # noqa: E402
from dl_02_dataset import WetlandPatchDataset  # noqa: E402

print(f"Project root: {PROJECT_ROOT}")
print(f"Models dir:   {MODELS_DIR}")
print(f"Patches dir:  {PATCHES_DIR}")

Project root: /ibstorage/anthony/NYS_Wetlands_DL
Models dir:   /ibstorage/anthony/NYS_Wetlands_DL/Models
Patches dir:  /ibstorage/anthony/NYS_Wetlands_DL/Data/Training_Data/R_Patches


## 2. Choose a checkpoint

The cell below lists every `.safetensors` / `.ckpt` in `Models/`. Set
`CHECKPOINT_NAME` to whichever you want — prefer the `.safetensors` (self-describing,
no architecture flags needed).

In [6]:
# --- Available checkpoints ---
checkpoints = sorted(
    {p.name for p in MODELS_DIR.glob("*.safetensors")}
    | {p.name for p in MODELS_DIR.glob("*.ckpt")}
)
print("Available checkpoints in Models/:")
for i, name in enumerate(checkpoints):
    print(f"  [{i}] {name}")

# --- Pick one (edit this line; .safetensors preferred) ---
CHECKPOINT_NAME = "best_multiclass_unet3plus_bf64_d4_20260605_1537.safetensors"
# Or select by index instead, e.g.:
# CHECKPOINT_NAME = checkpoints[0]

CHECKPOINT_PATH = MODELS_DIR / CHECKPOINT_NAME
assert CHECKPOINT_PATH.exists(), f"Not found: {CHECKPOINT_PATH}"
print(f"\nSelected: {CHECKPOINT_NAME}")

Available checkpoints in Models/:
  [0] best_binary_bf64_d4_20260421_1730.safetensors
  [1] best_binary_bf64_d4_20260421_1745.safetensors
  [2] best_binary_bf64_d4_20260511_1527.ckpt
  [3] best_binary_bf64_d4_20260511_1527.safetensors
  [4] best_binary_bf64_d4_20260515_2343.ckpt
  [5] best_binary_bf64_d4_20260515_2343.safetensors
  [6] best_binary_unet3plus_bf64_d4_20260605_1548.ckpt
  [7] best_binary_unet3plus_bf64_d4_20260605_1548.safetensors
  [8] best_multiclass_bf128_d4_20260420_1924.ckpt
  [9] best_multiclass_bf128_d4_20260420_1924.safetensors
  [10] best_multiclass_bf128_d4_20260420_2139.ckpt
  [11] best_multiclass_bf128_d4_20260420_2139.safetensors
  [12] best_multiclass_bf128_d5_20260420_2152.ckpt
  [13] best_multiclass_bf128_d5_20260420_2152.safetensors
  [14] best_multiclass_bf64_d4_20260420_0134.ckpt
  [15] best_multiclass_bf64_d4_20260420_0134.safetensors
  [16] best_multiclass_bf64_d4_20260420_0203.ckpt
  [17] best_multiclass_bf64_d4_20260420_0209.ckpt
  [18] best_multicl

## 3. Visualization settings

The knobs for everything below: how many channels to show per stage (`CHANNELS_PER_LEVEL`),
the heatmap color scheme (`FEATURE_CMAP`), how see-through the interactive map overlays are
(`OVERLAY_OPACITY`), and which hardware to run on (`DEVICE` — CPU is plenty for a single
patch; MPS/GPU also work).

In [7]:
# How many top-variance channels to show per level (1-3 recommended).
CHANNELS_PER_LEVEL = 4

# Colormap for feature-map heatmaps.
FEATURE_CMAP = "viridis"

# Opacity of the raster overlays on the interactive folium maps (0=invisible, 1=solid).
# Lower it (~0.6) to see more of the satellite basemap through the layers.
OVERLAY_OPACITY = 0.8

# Device: "mps" = Apple GPU. One patch is sub-second either way, so "cpu" works just as
# well (and is fully deterministic) — swap the string if MPS gives you any trouble.
DEVICE = torch.device("mps")
# from dl_03_unet_model import get_device; DEVICE = get_device()  # auto-pick CUDA/MPS/CPU
print(f"Device: {DEVICE}")

Device: mps


## 4. Load the model

`load_model()` reads the architecture straight from the `.safetensors` sidecar
(`.meta.json`) or the `.ckpt` hyperparameters, so no architecture flags are needed.

In [8]:
# in_channels / num_classes here are fallbacks only; for .safetensors/.ckpt they're
# overridden by the checkpoint metadata.
model = load_model(
    CHECKPOINT_PATH,
    device=DEVICE,
    in_channels=26,
    num_classes=4,
)
model.eval()

DEPTH = model.depth
NUM_CLASSES = model.num_classes
ARCH = type(model).__name__
print(f"\nArchitecture: {ARCH}, depth={DEPTH}, num_classes={NUM_CLASSES}")

# Class names + classification mode: prefer the checkpoint's .meta.json sidecar.
meta_path = CHECKPOINT_PATH.with_suffix(".meta.json")
if meta_path.exists():
    meta = json.loads(meta_path.read_text())
    CLASS_NAMES = meta.get("class_names", [f"class_{i}" for i in range(NUM_CLASSES)])
    MODE = meta.get("classification_mode", "multiclass")
else:
    CLASS_NAMES = [f"class_{i}" for i in range(NUM_CLASSES)]
    MODE = "multiclass"
print(f"Classes: {CLASS_NAMES}  (mode={MODE})")

ModuleNotFoundError: No module named 'safetensors'

## 5. Choose a patch & normalize it

Pick one 256 m × 256 m training patch to send through the model. Before the model sees it,
every band is **normalized** — rescaled to roughly 0–1 using the same statistics as during
training — so a band measured in big units (elevation in meters) can't drown out one that
lives between 0 and 1 (a vegetation fraction). We reuse the exact `WetlandPatchDataset`
code from training, so the model gets inputs in the form it learned on; the stats file is
matched to the checkpoint's classification mode.

One count to not be confused by: the patch has **18 raster bands** (17 predictors +
the MOD_CLASS label), but the model input below has **26 channels**. That's because the
categorical landform band (`Geomorph_local`) is **one-hot encoded** — split into ten
separate 0/1 layers, one per landform type — since landform category #7 isn't "more than"
#3; the categories have no numeric order a network could safely do math on.

In [ ]:
# --- Stats file (normalization is identical across weight-power variants) ---
stats_candidates = [
    STATS_DIR / f"{MODE}_normalization_stats.json",
    STATS_DIR / f"{MODE}_normalization_stats_wp0.5.json",
    STATS_DIR / "normalization_stats.json",
]
STATS_PATH = next((p for p in stats_candidates if p.exists()), None)
assert STATS_PATH is not None, f"No stats file found. Looked for: {stats_candidates}"
print(f"Stats: {STATS_PATH.name}")

# --- Pick a patch (edit PATCH_NAME, or use a random one) ---
patch_files = sorted(PATCHES_DIR.glob("*.tif"))
print(f"{len(patch_files)} patches available. First few:")
for p in patch_files[:3]:
    print(f"  {p.name}")

In [ ]:
PATCH_NAME = patch_files[471].name
#PATCH_NAME = "gps_cluster_225_huc_043001060201_patch_2_256m.tif"
# Random instead: PATCH_NAME = np.random.RandomState(0).choice([p.name for p in patch_files])
PATCH_PATH = PATCHES_DIR / PATCH_NAME
assert PATCH_PATH.exists(), f"Not found: {PATCH_PATH}"
print(f"\nSelected patch: {PATCH_NAME}")

In [ ]:
# Normalize the patch exactly as the training pipeline does.
ds = WetlandPatchDataset([PATCH_PATH], STATS_PATH, augment=False, validate_bands=False)
predictors, labels = ds[0]            # (in_channels, H, W), (H, W)
x = predictors.unsqueeze(0).to(DEVICE)  # add batch dim
labels = labels.numpy()
print(f"Input tensor: {tuple(x.shape)}, dtype={x.dtype}")
print(f"Label unique: {np.unique(labels).tolist()}  (255 = unlabeled)")

# Band names for reference (the folium map in section 8 reads the rasters itself).
with rasterio.open(PATCH_PATH) as src:
    band_names = list(src.descriptions)
print(f"Bands in patch: {band_names}")

## 6. Register forward hooks & run the patch

A **forward hook** is a small listener attached to a layer: when the patch flows through the
network, the hook quietly copies that layer's output so we can plot it afterward — the model
itself is never modified. We attach one to every encoder level, the bottleneck, every
decoder node, and the output head, then run a single forward pass.

- `encoders[i]` → returns `(pooled, skip)`; we keep the **skip** (the full feature map at level *i*)
- `bottleneck` → the coarsest, most abstract feature map
- `fuse_se[idx]` → a fused full-scale decoder node (UNet3+); for the plain UNet we fall back to its decoder blocks
- `head` → the **logits** (raw, pre-softmax class scores)

In the printout below, notice the trade the encoder makes: each level **halves** the image
size but **doubles** the channel count (64 → 128 → 256 → 512 → 1024) — giving up *where*
detail to store more kinds of *what* information. The decoder then runs the size back up
to 256×256, ending at the head's 4 channels: one score map per wetland class.

In [ ]:
features = {}   # name -> (C, H, W) numpy array for the single patch
hooks = []

def _cap(name, take_skip=False):
    def hook(module, inp, out):
        t = out[1] if take_skip else out
        features[name] = t.detach().cpu().float().numpy()[0]
    return hook

# Encoder levels (skip = pre-downsample feature at that level's resolution).
for i, enc in enumerate(model.encoders):
    hooks.append(enc.register_forward_hook(_cap(f"enc{i}", take_skip=True)))

# Bottleneck (captured pre-dropout; dropout is identity in eval anyway).
hooks.append(model.bottleneck.register_forward_hook(_cap("bottleneck")))

# Decoder nodes.
if hasattr(model, "fuse_se"):  # UNet3+
    for idx, lvl in enumerate(model._decoder_levels):
        hooks.append(model.fuse_se[idx].register_forward_hook(_cap(f"dec{lvl}")))
elif hasattr(model, "decoders"):  # plain UNet
    for idx, dec in enumerate(model.decoders):
        lvl = (DEPTH - 1) - idx
        hooks.append(dec.register_forward_hook(_cap(f"dec{lvl}")))

# Final logits.
head = model.head if hasattr(model, "head") else model.output
hooks.append(head.register_forward_hook(_cap("logits")))

# Forward pass (deep-supervision models return a single tensor in eval mode).
with torch.no_grad():
    out = model(x)
if isinstance(out, (list, tuple)):
    out = out[0]

for h in hooks:
    h.remove()

print("Captured feature maps (name: channels x H x W):")
for k, v in features.items():
    print(f"  {k:12s} {v.shape[0]:4d} x {v.shape[1]:3d} x {v.shape[2]:3d}")

## 7. Plot helpers

Small reusable functions for the plots that follow: `norm01` rescales a feature map to the
0–1 range so it displays well, `top_variance_channels` picks the most active channels, and
`plot_levels` lays out the per-stage grids. (Note: because each map is rescaled on its own,
brightness is only comparable *within* a single panel, not across panels.)

In [ ]:
def norm01(a):
    """Min-max a 2D map to [0,1] for display (constant maps -> zeros)."""
    a = np.nan_to_num(a.astype(np.float32))
    lo, hi = float(a.min()), float(a.max())
    return (a - lo) / (hi - lo) if hi > lo else np.zeros_like(a)

def top_variance_channels(fmap, k):
    """Indices of the k channels with the highest spatial variance."""
    var = fmap.reshape(fmap.shape[0], -1).var(axis=1)
    return np.argsort(var)[::-1][:k]

def plot_levels(level_keys, k=CHANNELS_PER_LEVEL, mode="variance", suptitle=""):
    """Grid of feature maps: one row per level.

    mode='variance' -> k top-variance channels per row.
    mode='mean'     -> single mean-activation map per row.
    """
    keys = [kk for kk in level_keys if kk in features]
    ncols = k if mode == "variance" else 1
    fig, axes = plt.subplots(len(keys), ncols, figsize=(3.0 * ncols, 3.0 * len(keys)),
                             squeeze=False)
    for r, key in enumerate(keys):
        fmap = features[key]
        C, H, W = fmap.shape
        if mode == "variance":
            for c, ci in enumerate(top_variance_channels(fmap, k)):
                ax = axes[r][c]
                ax.imshow(norm01(fmap[ci]), cmap=FEATURE_CMAP)
                ax.set_title(f"{key}  ch{ci}\n{C}ch @ {H}x{W}", fontsize=8)
                ax.axis("off")
        else:
            ax = axes[r][0]
            ax.imshow(norm01(fmap.mean(axis=0)), cmap=FEATURE_CMAP)
            ax.set_title(f"{key}  mean of {C}ch\n{H}x{W}", fontsize=8)
            ax.axis("off")
    if suptitle:
        fig.suptitle(suptitle, fontsize=12, y=1.0)
    fig.tight_layout()
    plt.show()

## 8. Input reference

Before looking inside the model, here's the context: the model's inputs and the *answer key*.
The interactive map below lets you flip through every input layer — the NAIP true-color
imagery (what a person would see), each individual predictor band, and the ground-truth
label map (MOD_CLASS) a human annotator assigned. Keep these in mind — every feature map
further down is the model working its way from those inputs toward that label map.

### 8b. Interactive Leaflet map (folium)

Every input layer rendered as a georeferenced overlay on an **Esri satellite** basemap.
This is a real web map — pan, zoom (scroll), and read coordinates just like in a GIS:

- The **radio control (top-right)** flips through layers one at a time: the **NAIP RGB
  (leaf-on)** and **Leaf-off RGB** true-color composites, the **MOD_CLASS label**
  (class colors, legend bottom-left), and each individual predictor band.
- **"None (basemap only)"** switches every overlay off so you can compare a band
  against the real ground in the satellite image.
- The **red outline** marks the patch footprint (256 m × 256 m) — zoom out and you can
  see exactly where in New York this patch sits.
- A **scale bar** (bottom-left) and the **cursor's lat/lon** (bottom-right) help relate
  what you see to ground distances and locations.

Continuous bands get a percentile (2–98%) contrast stretch; nodata/unlabeled pixels are
transparent. **Geomorph_local** is the exception — it's *categorical* (ten landform types,
not a measured quantity), so it's drawn with the standard geomorphon colors and its own
legend pops up (bottom-right) while that layer is selected. Geomorphons describe local
terrain shape from the DEM; watch how the water-collecting landforms — **valleys, hollows,
footslopes, pits, flats** — trace the drainage network where wetlands tend to form.

Overlays are embedded in the notebook (work offline); only the basemap tiles need internet.

In [ ]:
import io
import base64
from PIL import Image
import matplotlib
import folium
from folium.plugins import GroupedLayerControl, MousePosition
from rasterio.warp import transform_bounds

# Class colormap (extend the palette if you have >5 classes).
_palette = ["#2c7fb8", "#41b6c4", "#a1dab4", "#d95f0e", "#756bb1", "#636363"]
CLASS_CMAP = ListedColormap(_palette[:NUM_CLASSES])

def _legend_div(title, items, *, side="left", bottom=25, div_id=None, hidden=False):
    """Floating HTML legend box: items = [(css_color, label), ...]."""
    rows = "".join(
        f'<div style="margin:2px 0;">'
        f'<span style="display:inline-block;width:14px;height:14px;'
        f'background:{color};border:1px solid #555;'
        f'margin-right:6px;vertical-align:middle;"></span>'
        f'<span style="vertical-align:middle;">{label}</span></div>'
        for color, label in items
    )
    id_attr = f'id="{div_id}" ' if div_id else ""
    display = "display:none;" if hidden else ""
    return folium.Element(
        f'<div {id_attr}style="position:fixed;bottom:{bottom}px;{side}:12px;z-index:9999;'
        f'{display}background:rgba(255,255,255,0.9);padding:8px 10px;border:1px solid #888;'
        'border-radius:5px;font-size:12px;font-family:sans-serif;'
        'box-shadow:0 1px 4px rgba(0,0,0,0.3);">'
        f'<div style="font-weight:bold;margin-bottom:4px;">{title}</div>{rows}</div>'
    )

def class_legend_html(class_names, cmap, title="Wetland classes"):
    """Always-on wetland class legend; colors pulled straight from CLASS_CMAP so the
    legend always matches the class overlays (MOD_CLASS label, prediction, ground truth)."""
    from matplotlib.colors import to_hex
    return _legend_div(title, [(to_hex(cmap(i)), name) for i, name in enumerate(class_names)])

# ── Geomorphon landform classes ──
# Geomorph_local is CATEGORICAL: the standard 10 geomorphon landform types
# (Jasiewicz & Stepinski 2013), coded 1-10. Colors follow the conventional geomorphon
# palette — dark red crests, yellow slopes, blue drainage. For wetlands, the landforms
# where water collects (valley, hollow, footslope, pit, flat) matter most.
GEOMORPH_CLASSES = {
    1:  ("Flat — level ground",            "#dcdcdc"),
    2:  ("Peak — local high point",        "#380000"),
    3:  ("Ridge — narrow crest",           "#c80000"),
    4:  ("Shoulder — upper slope break",   "#ff5014"),
    5:  ("Spur — downhill-pointing ridge", "#fad23c"),
    6:  ("Slope — uniform hillside",       "#ffff3c"),
    7:  ("Hollow — upslope valley head",   "#b4e614"),
    8:  ("Footslope — base of hillside",   "#3cfa96"),
    9:  ("Valley — drainage bottom",       "#0000ff"),
    10: ("Pit — closed depression",        "#000038"),
}

def add_geomorph_legend(map_, geomorph_fg):
    """Geomorphon legend (bottom-right) that auto-shows only while the Geomorph_local
    layer is switched on — listens for that layer being added/removed from the map."""
    div_id = f"geomorph-legend-{map_.get_name()}"
    items = [(color, f"{v}: {name}") for v, (name, color) in GEOMORPH_CLASSES.items()]
    map_.get_root().html.add_child(
        _legend_div("Geomorphon landforms", items, side="right", bottom=45,
                    div_id=div_id, hidden=True))
    # IMPORTANT: folium renders custom script elements BEFORE the map's own JS, so the
    # map/layer variables don't exist yet at the point this code first runs. Registering
    # the handlers is therefore deferred until the document finishes loading — touching
    # the variables at the top level would throw and blank the entire map.
    hook = f"geomorphLegendHook_{map_.get_name()}"
    map_.get_root().script.add_child(folium.Element(f"""
        function {hook}() {{
            var legend = document.getElementById('{div_id}');
            {map_.get_name()}.on('layeradd', function(e) {{
                if (e.layer === {geomorph_fg.get_name()}) legend.style.display = 'block';
            }});
            {map_.get_name()}.on('layerremove', function(e) {{
                if (e.layer === {geomorph_fg.get_name()}) legend.style.display = 'none';
            }});
        }}
        if (document.readyState === 'loading') {{
            document.addEventListener('DOMContentLoaded', {hook});
        }} else {{
            {hook}();
        }}
    """))

# Per-band colormaps for CONTINUOUS bands (anything unlisted falls back to viridis;
# Geomorph_local is categorical and handled by GEOMORPH_CLASSES instead).
BAND_CMAPS = {
    "DEM": "gist_earth", "slope_local": "magma", "twi": "Blues",
    "flowacc": "Blues", "CHM": "YlGn",
    "pct_below_1m": "YlGn", "pct_1m_to_5m": "YlGn", "pct_above_5m": "YlGn",
}

def _stretch(c, nodata, pct):
    """Percentile-stretch one channel to [0,1]; return (values, nodata_mask)."""
    c = c.astype(np.float32)
    m = np.isnan(c)
    if nodata is not None and not np.isnan(nodata):
        m |= (c == nodata)
    valid = c[~m]
    lo, hi = (np.percentile(valid, pct) if valid.size else (0.0, 1.0))
    if hi <= lo:
        hi = lo + 1e-6
    return np.clip((c - lo) / (hi - lo), 0, 1), m

def _rgba_to_datauri(rgba):
    buf = io.BytesIO()
    Image.fromarray(rgba, mode="RGBA").save(buf, format="PNG")
    return "data:image/png;base64," + base64.b64encode(buf.getvalue()).decode()

def band_to_datauri(arr, cmap_name, nodata=None, pct=(2, 98)):
    """Single band -> colormapped base64 PNG (nodata transparent)."""
    norm, mask = _stretch(arr, nodata, pct)
    rgba = (matplotlib.colormaps[cmap_name](norm) * 255).astype(np.uint8)
    rgba[mask, 3] = 0
    return _rgba_to_datauri(rgba)

def rgb_to_datauri(r, g, b, nodata=None, pct=(2, 98)):
    """Three bands -> true-color base64 PNG (nodata in any channel transparent)."""
    chans, mask = [], None
    for c in (r, g, b):
        norm, m = _stretch(c, nodata, pct)
        chans.append(norm)
        mask = m if mask is None else (mask | m)
    rgb = (np.stack(chans, axis=-1) * 255).astype(np.uint8)
    alpha = np.where(mask, 0, 255).astype(np.uint8)
    rgba = np.dstack([rgb, alpha])
    return _rgba_to_datauri(rgba)

def label_to_datauri(arr, cmap, ignore=255):
    """Discrete class label -> class-colored base64 PNG (ignore_index transparent)."""
    a = arr.astype(np.float32)
    mask = np.isnan(a) | (a == ignore)
    idx = np.clip(np.where(mask, 0, np.nan_to_num(a)).astype(int), 0, cmap.N - 1)
    rgba = (cmap(idx) * 255).astype(np.uint8)
    rgba[mask, 3] = 0
    return _rgba_to_datauri(rgba)

def categorical_to_datauri(arr, value_colors):
    """Integer-coded categorical band -> base64 PNG with one fixed color per category
    (no stretch — category 7 is a kind, not a quantity). NaN/unknown values transparent."""
    a = arr.astype(np.float32)
    rgba = np.zeros((*a.shape, 4), dtype=np.uint8)
    for val, (_, hexcolor) in value_colors.items():
        rgb = tuple(int(hexcolor.lstrip("#")[i:i + 2], 16) for i in (0, 2, 4))
        rgba[a == val] = (*rgb, 255)
    return _rgba_to_datauri(rgba)

# Read every band + reproject the patch bounds to lat/lon for Leaflet.
with rasterio.open(PATCH_PATH) as src:
    all_bands = list(src.descriptions)
    nodata = src.nodata
    raster = src.read()
    west, south, east, north = transform_bounds(src.crs, "EPSG:4326", *src.bounds)

bounds_latlon = [[south, west], [north, east]]
LABEL_BAND = ds.label_band
get = lambda name: raster[all_bands.index(name)]  # noqa: E731
display_bands = [b for b in all_bands if b != LABEL_BAND]

# Build the ordered overlay list: RGB composites, label, then individual bands.
overlays = []  # (name, data_uri)
if all(b in all_bands for b in ("r", "g", "b")):
    overlays.append(("NAIP RGB (leaf-on)", rgb_to_datauri(get("r"), get("g"), get("b"), nodata)))
if all(b in all_bands for b in ("r_lo", "g_lo", "b_lo")):
    overlays.append(("Leaf-off RGB", rgb_to_datauri(get("r_lo"), get("g_lo"), get("b_lo"), nodata)))
if LABEL_BAND in all_bands:
    overlays.append((f"{LABEL_BAND} (label)", label_to_datauri(get(LABEL_BAND), CLASS_CMAP)))
for band in display_bands:
    if band == "Geomorph_local":
        overlays.append((band, categorical_to_datauri(get(band), GEOMORPH_CLASSES)))
    else:
        overlays.append((band, band_to_datauri(get(band), BAND_CMAPS.get(band, "viridis"), nodata)))

# Esri World Imagery satellite basemap.
ESRI_IMAGERY = (
    "https://server.arcgisonline.com/ArcGIS/rest/services/"
    "World_Imagery/MapServer/tile/{z}/{y}/{x}"
)
m = folium.Map(
    location=[(south + north) / 2, (west + east) / 2],
    zoom_start=15,
    tiles=ESRI_IMAGERY,
    attr="Esri World Imagery",
    control_scale=True,  # metric scale bar, bottom-left
)

# One FeatureGroup per overlay; GroupedLayerControl makes them an exclusive (radio) set.
# A leading "None" option lets you switch every overlay OFF and view the bare basemap —
# radio groups always keep one option selected, so an empty group is how you "turn it off".
groups = []
input_fgs = {}  # name -> FeatureGroup (kept so legends can react to specific layers)
none_fg = folium.FeatureGroup(name="None (basemap only)", show=False)
none_fg.add_to(m)
groups.append(none_fg)
for i, (name, uri) in enumerate(overlays):
    fg = folium.FeatureGroup(name=name, show=(i == 0))
    folium.raster_layers.ImageOverlay(image=uri, bounds=bounds_latlon,
                                      opacity=OVERLAY_OPACITY).add_to(fg)
    fg.add_to(m)
    groups.append(fg)
    input_fgs[name] = fg

GroupedLayerControl(
    groups={"Input layers": groups},
    exclusive_groups=True,
    collapsed=True,
).add_to(m)

# Patch footprint outline (hover it for the size) + live cursor coordinates.
folium.Rectangle(bounds_latlon, color="#ff4444", weight=2, fill=False,
                 tooltip="Patch footprint (256 m × 256 m)").add_to(m)
MousePosition(position="bottomright", prefix="lat/lon:", num_digits=5).add_to(m)

# Always-on class legend + geomorphon legend that appears with the Geomorph_local layer.
m.get_root().html.add_child(class_legend_html(CLASS_NAMES, CLASS_CMAP))
if "Geomorph_local" in input_fgs:
    add_geomorph_legend(m, input_fgs["Geomorph_local"])
m.fit_bounds(bounds_latlon)
print(f"Built Leaflet map with {len(overlays)} layers (+ 'None' toggle): {[n for n, _ in overlays]}")
m

## 9. Encoder feature maps (top-variance channels)

The encoder as it *zooms out*. The top row (`enc0`, full resolution) responds to low-level
cues — edges, texture, color/brightness contrast that still look a lot like the inputs.
Each row down is half the size and a step more abstract, trading spatial detail for
"meaning." Rows go fine → coarse. (We show the few most active channels at each level;
there are dozens to hundreds in total.)

*Tip:* flip the map above to **NAIP RGB** and compare — in `enc0`/`enc1` you can usually
recognize what a channel locked onto (a stream channel, a field edge, canopy texture).
By `enc3` that's much harder; the maps are responding to broader patterns, not objects.

In [ ]:
enc_keys = [f"enc{i}" for i in range(DEPTH)]
plot_levels(enc_keys, mode="variance", suptitle="Encoder — top-variance channels")

## 10. Bottleneck — BOTH views

The deepest, most compressed stage — the model's most abstract "summary" of the patch. It's
small and blurry *by design*: at this point the network captures roughly *what is here*
rather than *exactly where*. At depth 4 each bottleneck pixel summarizes a **16 m × 16 m**
area of ground — enough to say "wet meadow around here," but not where its edge runs;
re-drawing those edges is the decoder's job. Because this is the widest level (the most
channels), we show it two ways: the **top-variance channels** and the **mean of all
channels**, side by side.

In [ ]:
bn = features["bottleneck"]
C, H, W = bn.shape
k = CHANNELS_PER_LEVEL
fig, axes = plt.subplots(1, k + 1, figsize=(3.0 * (k + 1), 3.2), squeeze=False)

# Top-variance channels.
for c, ci in enumerate(top_variance_channels(bn, k)):
    axes[0][c].imshow(norm01(bn[ci]), cmap=FEATURE_CMAP)
    axes[0][c].set_title(f"bottleneck ch{ci}\n(top-var)", fontsize=9)
    axes[0][c].axis("off")

# Mean activation across all channels.
axes[0][k].imshow(norm01(bn.mean(axis=0)), cmap=FEATURE_CMAP)
axes[0][k].set_title(f"bottleneck mean\nof {C}ch", fontsize=9)
axes[0][k].axis("off")

fig.suptitle(f"Bottleneck — {C} channels @ {H}x{W}", fontsize=12, y=1.05)
fig.tight_layout()
plt.show()

## 11. Decoder feature maps (top-variance channels)

The decoder *zooming back in*. Each stage merges the abstract bottleneck summary with the
matching fine detail from the encoder (the "skip connections"), progressively sharpening
edges and re-localizing the wetland boundaries. Rows go coarse → fine; `dec0` is the
full-resolution node that feeds the final output head.

In [ ]:
dec_keys = [f"dec{i}" for i in range(DEPTH - 1, -1, -1)]
plot_levels(dec_keys, mode="variance", suptitle="Decoder — top-variance channels")

## 12. Whole-network mean-activation overview

One mean-activation image per stage, in order: encoder → bottleneck → decoder. This is the
"signal flow" at a glance — watch the resolution shrink through the encoder, bottom out at
the bottleneck, then grow back through the decoder, getting reorganized around the wetland
features along the way.

In [ ]:
overview_keys = (
    [f"enc{i}" for i in range(DEPTH)]
    + ["bottleneck"]
    + [f"dec{i}" for i in range(DEPTH - 1, -1, -1)]
)
overview_keys = [k for k in overview_keys if k in features]
ncols = 5
nrows = int(np.ceil(len(overview_keys) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(3.0 * ncols, 3.0 * nrows), squeeze=False)
for ax in axes.ravel():
    ax.axis("off")
for idx, key in enumerate(overview_keys):
    fmap = features[key]
    ax = axes[idx // ncols][idx % ncols]
    ax.imshow(norm01(fmap.mean(axis=0)), cmap=FEATURE_CMAP)
    ax.set_title(f"{key}\n{fmap.shape[0]}ch @ {fmap.shape[1]}x{fmap.shape[2]}", fontsize=8)
    ax.axis("off")
fig.suptitle("Mean activation per level (encoder -> bottleneck -> decoder)", fontsize=12, y=1.0)
fig.tight_layout()
plt.show()

## 13. Logits → predicted classes

The payoff. The output head turns the final decoder feature map into a raw score (a
**logit**) for each class at each pixel. **Softmax** converts those scores into
probabilities that sum to 1, and **argmax** picks the highest-probability class as the
prediction. Shown here: a probability heatmap per class, the predicted class map, and the
ground truth for comparison.

In [ ]:
probs = torch.softmax(out, dim=1)[0].cpu().numpy()  # (num_classes, H, W)
pred = probs.argmax(axis=0)

# Per-class probability heatmaps.
fig, axes = plt.subplots(1, NUM_CLASSES, figsize=(3.2 * NUM_CLASSES, 3.4), squeeze=False)
for c in range(NUM_CLASSES):
    im = axes[0][c].imshow(probs[c], cmap="magma", vmin=0, vmax=1)
    axes[0][c].set_title(f"P({CLASS_NAMES[c]})", fontsize=10)
    axes[0][c].axis("off")
    fig.colorbar(im, ax=axes[0][c], fraction=0.046)
fig.suptitle("Per-class softmax probability", fontsize=12, y=1.04)
fig.tight_layout()
plt.show()

# Prediction vs. ground truth.
fig, axes = plt.subplots(1, 2, figsize=(11, 5))
im0 = axes[0].imshow(pred, cmap=CLASS_CMAP, vmin=0, vmax=NUM_CLASSES - 1)
axes[0].set_title("Predicted class (argmax)")
axes[0].axis("off")
axes[1].imshow(np.ma.masked_equal(labels, 255), cmap=CLASS_CMAP, vmin=0, vmax=NUM_CLASSES - 1)
axes[1].set_title("Ground truth (MOD_CLASS)")
axes[1].axis("off")
cbar = fig.colorbar(im0, ax=axes, fraction=0.046, ticks=range(NUM_CLASSES))
cbar.ax.set_yticklabels(CLASS_NAMES)
plt.show()

## 14. Deep-supervision side predictions (per-stage, coarse → fine)

**Deep supervision** means that during training the model wasn't graded only on its final
full-resolution map — every decoder stage *and* the bottleneck got its own tiny prediction
head and was graded on a "rough draft" of the wetland map at its own scale. That pressure
pushes *every* level of the network to learn wetland-relevant features, not just the last one.

At inference those side heads are normally ignored (`eval` mode returns only the main,
finest prediction), but they were saved with the model — so here we apply each one to the
feature maps we captured in section 6 and watch the prediction sharpen from the blurry
16×16 bottleneck draft to the full-resolution final map. Top row = predicted class;
bottom row = model confidence (the winning class's probability). The printout at the end
quantifies how much each draft already agrees with the final answer.

In [ ]:
import torch.nn.functional as F

if getattr(model, "deep_supervision", False) and hasattr(model, "ds_heads"):
    full = tuple(out.shape[-2:])

    # Apply each trained side head to its captured feature map, then upsample to full res.
    stage_logits = {}
    with torch.no_grad():
        bn = torch.from_numpy(features["bottleneck"]).unsqueeze(0).to(DEVICE)
        stage_logits["bottleneck"] = F.interpolate(
            model.ds_heads["bottleneck"](bn), size=full, mode="bilinear", align_corners=False)
        for i in range(model.depth - 1, 0, -1):  # dec3, dec2, dec1
            feat = torch.from_numpy(features[f"dec{i}"]).unsqueeze(0).to(DEVICE)
            stage_logits[f"dec{i}"] = F.interpolate(
                model.ds_heads[str(i)](feat), size=full, mode="bilinear", align_corners=False)
    stage_logits["dec0 (main)"] = out  # main head, already full resolution

    order = ["bottleneck"] + [f"dec{i}" for i in range(model.depth - 1, 0, -1)] + ["dec0 (main)"]
    ncols = len(order)
    fig, axes = plt.subplots(2, ncols, figsize=(2.9 * ncols, 6.2), squeeze=False)
    for c, key in enumerate(order):
        probs = torch.softmax(stage_logits[key], dim=1)[0].cpu().numpy()
        im_cls = axes[0][c].imshow(probs.argmax(0), cmap=CLASS_CMAP, vmin=0, vmax=NUM_CLASSES - 1)
        axes[0][c].set_title(f"{key}\nargmax class", fontsize=9)
        axes[0][c].axis("off")
        im_conf = axes[1][c].imshow(probs.max(0), cmap="magma", vmin=0, vmax=1)
        axes[1][c].set_title("max prob (confidence)", fontsize=9)
        axes[1][c].axis("off")

    cbar = fig.colorbar(im_cls, ax=axes[0].tolist(), fraction=0.025, ticks=range(NUM_CLASSES))
    cbar.ax.set_yticklabels(CLASS_NAMES)
    fig.colorbar(im_conf, ax=axes[1].tolist(), fraction=0.025)
    fig.suptitle("Deep-supervision side predictions: coarse (bottleneck) -> fine (main head)",
                 fontsize=12, y=1.02)
    plt.show()

    # Agreement of each side head with the final main prediction.
    main_pred = torch.softmax(out, dim=1)[0].cpu().numpy().argmax(0)
    print("Per-stage agreement with the main head's prediction:")
    for key in order:
        pred = torch.softmax(stage_logits[key], dim=1)[0].cpu().numpy().argmax(0)
        print(f"  {key:14s} {(pred == main_pred).mean() * 100:5.1f}% of pixels match")
else:
    print("This model was not trained with deep supervision \u2014 no per-stage heads to visualize.")
    print("(Only the single main output head exists; see section 13.)")

## 15. Combined interactive map — predictions + inputs (folium)

The same Esri-satellite Leaflet map as section 8b, now with the **model's outputs** added —
flip between *what the model saw* and *what it predicted*, all draped over real imagery.
The layer control (top-right) now has **two independent radio groups**, so you can show
one **model output** and one **input layer** at the same time — the output always draws on
top (try *prediction confidence* over *NAIP RGB* to see what the model hesitates about):

- **+ Predicted class** (argmax) vs. **+ Ground truth** (MOD_CLASS) — flip back and forth to spot errors
- **+ Prediction confidence** (max probability) and a **+ P(class)** layer per class — where the model is sure vs. unsure
- all the NAIP / leaf-off composites and individual input bands from section 8b

Each group has a **"None"** option — pick None in both to see the bare satellite image.
This is the one map to share if you want someone to explore the model interactively.

In [ ]:
# ── Combined interactive map: predictions + ground truth + all input layers ──
# Reuses the helper functions, Esri basemap, bounds, and `overlays` list from section 8b
# (run that cell first). Predictions are recomputed from `out` so this cell is independent
# of the loop variables in section 14.
pred_probs = torch.softmax(out, dim=1)[0].cpu().numpy()   # (num_classes, H, W)
pred_cls = pred_probs.argmax(0)                            # winning class per pixel

def prob_to_datauri(p, cmap_name="magma"):
    """Probability map already in [0,1] -> base64 PNG (no stretch; absolute 0-1 scale)."""
    rgba = (matplotlib.colormaps[cmap_name](np.clip(p, 0, 1)) * 255).astype(np.uint8)
    return _rgba_to_datauri(rgba)

# Model-output overlays (marked with a leading "+" so they read as a group).
pred_overlays = [
    ("+ Predicted class (argmax)", label_to_datauri(pred_cls.astype(np.float32), CLASS_CMAP)),
    ("+ Ground truth (MOD_CLASS)", label_to_datauri(get(LABEL_BAND), CLASS_CMAP)),
    ("+ Prediction confidence (max prob)", prob_to_datauri(pred_probs.max(0))),
]
for c in range(NUM_CLASSES):
    pred_overlays.append((f"+ P({CLASS_NAMES[c]})", prob_to_datauri(pred_probs[c])))

m2 = folium.Map(
    location=[(south + north) / 2, (west + east) / 2],
    zoom_start=15,
    tiles=ESRI_IMAGERY,
    attr="Esri World Imagery",
    control_scale=True,
)

def radio_group(map_, layer_list, default_name=None, z_index=1):
    """One FeatureGroup per overlay, plus a leading empty 'None' group.

    GroupedLayerControl turns the list into a radio set, and radio sets always keep one
    option selected — the empty 'None' group is how you switch the whole set off.
    z_index keeps model outputs drawn ABOVE input layers when one of each is shown.
    """
    groups = [folium.FeatureGroup(name="None", show=(default_name is None)).add_to(map_)]
    for name, uri in layer_list:
        fg = folium.FeatureGroup(name=name, show=(name == default_name))
        folium.raster_layers.ImageOverlay(
            image=uri, bounds=bounds_latlon, opacity=OVERLAY_OPACITY, z_index=z_index,
        ).add_to(fg)
        fg.add_to(map_)
        groups.append(fg)
    return groups

# Two independent radio sets: pick one model output AND one input layer at a time.
input_groups = radio_group(m2, overlays, default_name=None, z_index=1)
output_groups = radio_group(m2, pred_overlays,
                            default_name="+ Predicted class (argmax)", z_index=10)

GroupedLayerControl(
    groups={"Model outputs": output_groups, "Input layers": input_groups},
    exclusive_groups=True,
    collapsed=True,
).add_to(m2)

# Patch outline, cursor coordinates, and the class legend (matches the class overlays).
folium.Rectangle(bounds_latlon, color="#ff4444", weight=2, fill=False,
                 tooltip="Patch footprint (256 m × 256 m)").add_to(m2)
MousePosition(position="bottomright", prefix="lat/lon:", num_digits=5).add_to(m2)
m2.get_root().html.add_child(class_legend_html(CLASS_NAMES, CLASS_CMAP))

# Geomorphon landform legend, shown only while the Geomorph_local layer is on.
geo_fg = next((g for g in input_groups if getattr(g, "layer_name", None) == "Geomorph_local"), None)
if geo_fg is not None:
    add_geomorph_legend(m2, geo_fg)

m2.fit_bounds(bounds_latlon)
print(f"Built combined map: {len(pred_overlays)} model-output layers + {len(overlays)} "
      f"input layers, in two radio groups (each with its own 'None' option).")
m2

## 16. Side-by-side error check — prediction vs. ground truth (synced maps)

Two maps locked together: **left = the model's prediction**, **right = the analyst's
ground truth** (unlabeled pixels are transparent, letting the imagery show through).
Pan or zoom either side and the other follows, so your eyes can stay on one spot while
you compare. This is the fastest way to see *where* the model goes wrong — a marsh edge
drawn too wide, an upland inclusion it missed. Same class colors as the legend
(bottom-left).

In [ ]:
# ── Synced side-by-side comparison (folium DualMap) ──
from folium.plugins import DualMap

m3 = DualMap(
    location=[(south + north) / 2, (west + east) / 2],
    zoom_start=16,
    tiles=ESRI_IMAGERY,
    attr="Esri World Imagery",
    control_scale=True,
)

# Left map (m1): model prediction. Right map (m2): ground truth (unlabeled transparent).
folium.raster_layers.ImageOverlay(
    image=label_to_datauri(pred_cls.astype(np.float32), CLASS_CMAP),
    bounds=bounds_latlon, opacity=OVERLAY_OPACITY,
).add_to(m3.m1)
folium.raster_layers.ImageOverlay(
    image=label_to_datauri(get(LABEL_BAND), CLASS_CMAP),
    bounds=bounds_latlon, opacity=OVERLAY_OPACITY,
).add_to(m3.m2)

# Patch outline on both sides (anything added to m3 itself goes to both maps).
folium.Rectangle(bounds_latlon, color="#ff4444", weight=2, fill=False).add_to(m3)

# Floating titles so you don't lose track of which side is which, + shared legend.
for left_pct, text in (("25%", "Model prediction"), ("75%", "Ground truth (MOD_CLASS)")):
    m3.get_root().html.add_child(folium.Element(
        f'<div style="position:fixed;top:12px;left:{left_pct};transform:translateX(-50%);'
        'z-index:9999;background:rgba(255,255,255,0.92);padding:4px 10px;'
        'border:1px solid #888;border-radius:4px;font:bold 13px sans-serif;">'
        f'{text}</div>'
    ))
m3.get_root().html.add_child(class_legend_html(CLASS_NAMES, CLASS_CMAP))
m3.m1.fit_bounds(bounds_latlon)
m3.m2.fit_bounds(bounds_latlon)
m3

---
**To explore another patch or model:** change `CHECKPOINT_NAME` (Section 2) or
`PATCH_NAME` (Section 5), then re-run from that cell down. `CHANNELS_PER_LEVEL`,
`FEATURE_CMAP`, and `OVERLAY_OPACITY` (Section 3) control how the plots and the
interactive maps (Sections 8b, 15, 16) are drawn.